<a href="https://colab.research.google.com/github/upgrade-projects/pysparks-projects/blob/main/Global_Student_Migration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Global Student Migration and Higher Education Trends**

You are provided with a data set titled ‘Global Student Migration and Higher Education Trends’, which captures information about international students, including their country of origin, destination university, field of study, scholarships, post-graduation placements and language proficiency.   

With the given [data set](https://drive.google.com/file/d/1EGSCMLSkcT6yHn7hxP_J0c4J9pSLk9lb/view?usp=drive_link), solve the following tasks:

## **Setup: Mount Google Drive, Initialize SparkSession and Load Data**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install pyspark
!pip install pyspark

# Import SparkSession
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("GlobalStudentMigration").getOrCreate()

print("SparkSession initialized.")

SparkSession initialized.


In [ ]:
# Load the dataset
dataset_path = '/content/drive/MyDrive/Colab_Notebooks/Upgrad_Training/data/global_student_migration.csv'
df = spark.read.csv(dataset_path, header=True, inferSchema=True)

print("Dataset loaded successfully.")
print("Schema:")
df.printSchema()
print("First 5 rows:")
df.show(5)

Dataset loaded successfully.
Schema:
root
 |-- student_id: string (nullable = true)
 |-- origin_country: string (nullable = true)
 |-- destination_country: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- university_name: string (nullable = true)
 |-- course_name: string (nullable = true)
 |-- field_of_study: string (nullable = true)
 |-- year_of_enrollment: integer (nullable = true)
 |-- scholarship_received: string (nullable = true)
 |-- enrollment_reason: string (nullable = true)
 |-- graduation_year: integer (nullable = true)
 |-- placement_status: string (nullable = true)
 |-- placement_country: string (nullable = true)
 |-- placement_company: string (nullable = true)
 |-- starting_salary_usd: integer (nullable = true)
 |-- gpa_or_score: double (nullable = true)
 |-- visa_status: string (nullable = true)
 |-- post_graduation_visa: string (nullable = true)
 |-- language_proficiency_test: string (nullable = true)
 |-- test_score: double (nullable = true)

---

## **Task 1: Write a function that returns the number of students who received a scholarship.**

In [ ]:
from pyspark.sql.functions import col

def get_num_students_with_scholarship(dataframe):
    """
    Returns the number of students who received a scholarship.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame.

    Returns:
        int: The number of students with a scholarship.
    """
    # Inspect unique values in 'scholarship_received' column to determine the filter condition
    print("Unique values in 'scholarship_received' column:")
    dataframe.select('scholarship_received').distinct().show()

    scholarship_count = dataframe.filter(col('scholarship_received').cast('string').like('Yes')).count()

    return scholarship_count

# Demonstrate the function
num_students = get_num_students_with_scholarship(df)
print(f"Number of students who received a scholarship: {num_students}")

Unique values in 'scholarship_received' column:
+--------------------+
|scholarship_received|
+--------------------+
|                  No|
|                 Yes|
+--------------------+

Number of students who received a scholarship: 2577


---

## **Task 2: Write a function that returns the average GPA of students placed in the United States.**

In [ ]:
from pyspark.sql.functions import avg, col

def get_average_gpa_us_placement(dataframe):
    """
    Returns the average GPA of students placed in the United States.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame.

    Returns:
        float: The average GPA of students placed in the United States, or None if no data.
    """
    us_students_with_gpa = dataframe.filter(col('placement_country') == 'United States')\
                                     .filter(col('gpa_or_score').isNotNull())

    if us_students_with_gpa.count() > 0: # Check if there are any eligible students
        us_gpa = us_students_with_gpa.select(avg('gpa_or_score')).collect()[0][0]
        return us_gpa
    else:
        return None

# Demonstrate the function
avg_gpa_us = get_average_gpa_us_placement(df)
if avg_gpa_us is not None:
    print(f"Average GPA of students placed in the United States: {avg_gpa_us:.2f}")
else:
    print("No students found placed in the United States with a valid GPA.")

No students found placed in the United States with a valid GPA.


---

## **Task 3: Write a function that returns the top 5 destination countries for a given year of enrolment.**

In [ ]:
from pyspark.sql.functions import desc

def get_top_5_destination_countries_by_year(dataframe, year):
    """
    Returns the top 5 destination countries for a given year of enrolment.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame.
        year (int): The enrolment year.

    Returns:
        list: A list of the top 5 destination countries and their counts.
    """
    top_countries = dataframe.filter(col('year_of_enrollment') == year)\
                             .groupBy('destination_country')\
                             .count()\
                             .orderBy(desc('count'))\
                             .limit(5)
    return top_countries.collect()

# Demonstrate the function for a sample year (e.g., 2020)
sample_year = 2020
top_destination_countries = get_top_5_destination_countries_by_year(df, sample_year)
print(f"Top 5 destination countries for {sample_year}:")
for row in top_destination_countries:
    print(f"- {row['destination_country']}: {row['count']} students")

Top 5 destination countries for 2020:
- Ireland: 107 students
- UK: 107 students
- Germany: 105 students
- Finland: 101 students
- USA: 100 students


---

## **Task 4: Write a function that calculates the average starting salary for students who studied at universities with ‘Technology’ or ‘Engineering’ in their name and got placed.**

In [ ]:
from pyspark.sql.functions import lower, avg

def get_avg_salary_tech_eng_universities(dataframe):
    """
    Calculates the average starting salary for students who studied at universities
    with ‘Technology’ or ‘Engineering’ in their name and got placed.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame.

    Returns:
        float: The average starting salary.
    """
    avg_salary = dataframe.filter((lower(col('university_name')).contains('technology')) | \
                                  (lower(col('university_name')).contains('engineering')))\
                          .filter(col('placement_status') == 'Placed') \
                          .select(avg('starting_salary_usd')).collect()[0][0]
    return avg_salary

# Demonstrate the function
avg_salary = get_avg_salary_tech_eng_universities(df)
print(f"Average starting salary for students from 'Technology' or 'Engineering' universities who were placed: {avg_salary:.2f}")

Average starting salary for students from 'Technology' or 'Engineering' universities who were placed: 93310.03


---

## **Task 5: Write a function that returns the average TOEFL score by origin country for students who were placed in a different country and took the TOEFL test.**

In [ ]:
from pyspark.sql.functions import when, col, avg, desc

def get_avg_toefl_by_origin_country_placed_abroad(dataframe):
    """
    Returns the average TOEFL score by origin country for students who were placed
    in a different country than their origin and took the TOEFL test.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame.

    Returns:
        list: A list of rows containing origin country and their average TOEFL scores.
    """
    # Filter for students where Origin_Country is not equal to placement_country
    # and test_score is not null and is greater than 0 (assuming 0 is invalid)
    avg_toefl_scores = dataframe.filter((col('origin_country') != col('placement_country')) & \
                                        (col('test_score').isNotNull()) & \
                                        (col('test_score') > 0))\
                                  .groupBy('origin_country')\
                                  .agg(avg('test_score').alias('Average_TOEFL_Score'))\
                                  .orderBy(desc('Average_TOEFL_Score'))
    return avg_toefl_scores.collect()

# Demonstrate the function
avg_toefl_by_country = get_avg_toefl_by_origin_country_placed_abroad(df)
print("Average TOEFL score by origin country for students placed in a different country:")
for row in avg_toefl_by_country:
    print(f"- {row['origin_country']}: {row['Average_TOEFL_Score']:.2f}")

Average TOEFL score by origin country for students placed in a different country:
- Canada: 7.13
- Germany: 7.10
- India: 7.08
- Finland: 7.03
- UAE: 7.02
- Ireland: 6.97
- UK: 6.95
- Russia: 6.94
- USA: 6.94
- South Africa: 6.92
